# Phase 3 · Water quality prediction

Evaluate the saved XGBoost, Random Forest and MLP predictions of `CCME_Values`.
The AML recap comes from the original Stage 6 report. The side-by-side evaluation
uses the saved AML bridge exports and the matching DL prediction files.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import Image, display

cwd = Path.cwd()
repo = next(path for path in (cwd, *cwd.parents)
            if (path / "dl" / "configs" / "config.yaml").exists())
sys.path.insert(0, str(repo))
from dl.src.reporting import artifact_paths, run

# A scratch directory keeps notebook experiments away from checked-in results.
from tempfile import mkdtemp
output = Path(mkdtemp(prefix="capstone-notebook-"))
artifact_root = os.environ.get("AML_CAPSTONE_ARTIFACTS")
inputs = artifact_paths(Path(artifact_root)) if artifact_root else {}
status = run(output, **inputs)
status

## Point prediction

The single deep ensemble contains five trained members. Its exported aliases share
the same predictions. The other three full-feature DL variants report the mean and
sample standard deviation across five runs. AML metrics are imported.

In [ ]:
def table(name):
    return pd.read_csv(output / "tables" / f"phase3_{name}.csv")

def figure(name):
    display(Image(filename=str(output / "figures" / f"phase3_{name}.png"), width=900))

display(table("aml_stage6_recap"))
display(table("comparison").round(4))
display(table("resources"))

## Paired error diagnostics

Positive deltas mean higher RMSE for DL. The bootstrap resamples the same
observations for all models. With verified per-row site metadata it resamples whole
sites. The row-level fallback is exploratory because it assumes independent rows.
The CCME bands here use the study's declared class scheme.

In [ ]:
display(table("paired_bootstrap").round(4))
groups = table("subgroups")
columns = ["model", "seed", "group", "n", "RMSE", "MAE"]
display(groups.loc[groups.grouping == "CCME band", columns].round(4))
if status["metadata_available"]:
    display(groups.loc[groups.grouping != "CCME band", ["grouping", *columns]].round(4))
    display(table("sites").head(12))
figure("residuals")

## Uncertainty

PICP is the fraction of observations inside the nominal interval. MPIW is its mean
width in WQI points. XGBoost's conformal radius comes from saved validation
residuals. The quantile plot shows the effect of including or excluding ties at the
100-point ceiling. The regression runs export hard class tiers and omit class probabilities.

In [ ]:
display(table("intervals").round(4))
display(table("quantile_calibration").round(4))
figure("calibration")

## Explainability and the feature ablation

The original AML TreeSHAP figure and permutation scores are archived. The MLP diagnostics use saved weights and cleaned features when the project archive is supplied. The ablation
removes four engineered proxies. Raw chemistry can still carry the inputs used
in the index, so the ablation estimates sensitivity to those four proxies.

In [ ]:
display(table("aml_frozen_importance"))
display(Image(filename=str(repo / "outputs/figures/fig12_shap_summary.png"), width=720))
explanation = json.loads((output / "tables/phase3_explain_status.json").read_text())
if explanation["mlp_checkpoint_evaluated"]:
    display(explanation, table("mlp_importance"), table("permutation_overlap"))
    figure("mlp_shap")
    figure("mlp_permutation")
    figure("mlp_pdp_ice")
display(table("leakage_ablation").round(4))

## Interpretation

The split is site-held-out and uses the Phase 1 manifest. The provenance output above
records whether the original AML split archive was independently checked. The
comparison addresses reconstruction of the supplied CCME index. It does not establish
future forecasting performance or independence from the chemistry used in the index.